### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [14]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [16]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            # Add the documents to the list of all documents
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 1. Introduction.pdf
  ✓ Loaded 31 pages

Processing: 2. Organizations.pdf
  ✓ Loaded 28 pages

Processing: 3. Process Groups and Methodologies.pdf
  ✓ Loaded 42 pages

Processing: 4. Integration  Management.pdf
  ✓ Loaded 32 pages

Total documents loaded: 133


In [17]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-06-01T11:24:39+05:30', 'author': 'Dell', 'moddate': "D:20240607211157Z00'00", 'title': 'Project Management', 'source': '..\\data\\pdf\\1. Introduction.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': '1. Introduction.pdf', 'file_type': 'pdf'}, page_content='Project Management\nSENG 31232\nNabeel Rizwan\nBSc (Hons) Software Engineering'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-06-01T11:24:39+05:30', 'author': 'Dell', 'moddate': "D:20240607211157Z00'00", 'title': 'Project Management', 'source': '..\\data\\pdf\\1. Introduction.pdf', 'total_pages': 31, 'page': 1, 'page_label': '2', 'source_file': '1. Introduction.pdf', 'file_type': 'pdf'}, page_content='Learning Outcomes\n•Assess the differences between IT/IS projects and general projects\n•Explain how to in

In [18]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [19]:
chunks=split_documents(all_pdf_documents)
chunks

Split 133 documents into 133 chunks

Example chunk:
Content: Project Management
SENG 31232
Nabeel Rizwan
BSc (Hons) Software Engineering...
Metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-06-01T11:24:39+05:30', 'author': 'Dell', 'moddate': "D:20240607211157Z00'00", 'title': 'Project Management', 'source': '..\\data\\pdf\\1. Introduction.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': '1. Introduction.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-06-01T11:24:39+05:30', 'author': 'Dell', 'moddate': "D:20240607211157Z00'00", 'title': 'Project Management', 'source': '..\\data\\pdf\\1. Introduction.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': '1. Introduction.pdf', 'file_type': 'pdf'}, page_content='Project Management\nSENG 31232\nNabeel Rizwan\nBSc (Hons) Software Engineering'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-06-01T11:24:39+05:30', 'author': 'Dell', 'moddate': "D:20240607211157Z00'00", 'title': 'Project Management', 'source': '..\\data\\pdf\\1. Introduction.pdf', 'total_pages': 31, 'page': 1, 'page_label': '2', 'source_file': '1. Introduction.pdf', 'file_type': 'pdf'}, page_content='Learning Outcomes\n•Assess the differences between IT/IS projects and general projects\n•Explain how to in

### embedding And vectorStoreDB

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


d:\ml\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tharsikan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\ml\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-s

Model loaded successfully. Embedding dimension: 384


C:\Users\Tharsikan\AppData\Local\Temp\ipykernel_20756\1533322919.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [22]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 133 texts...


Batches: 100%|██████████| 5/5 [00:01<00:00,  2.83it/s]


Generated embeddings with shape: (133, 384)
Adding 133 documents to vector store...
Successfully added 133 documents to vector store
Total documents in collection: 133


### Retriever Pipeline From VectorStore

In [25]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                 print("No documents found")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [26]:
rag_retriever

In [27]:
rag_retriever.retrieve("What is a Project?")

Retrieving documents for query: 'What is a Project?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4615ed3e_4',
  'content': 'What is a Project? \nA project is “a temporary effort undertaken to create \na unique product, service, or result” \n(PMBOK® Guide, Sixth Edition, 2017)',
  'metadata': {'producer': 'Microsoft® PowerPoint® 2016',
   'source_file': '1. Introduction.pdf',
   'file_type': 'pdf',
   'page': 4,
   'page_label': '5',
   'total_pages': 31,
   'creator': 'Microsoft® PowerPoint® 2016',
   'doc_index': 4,
   'content_length': 149,
   'author': 'Dell',
   'title': 'Project Management',
   'moddate': "D:20240607211157Z00'00",
   'source': '..\\data\\pdf\\1. Introduction.pdf',
   'creationdate': '2024-06-01T11:24:39+05:30'},
  'similarity_score': 0.6227028965950012,
  'distance': 0.3772971034049988,
  'rank': 1},
 {'id': 'doc_17c00df7_6',
  'content': 'What is a Project? \n1. Unique - A project has a unique purpose. Every project should have a well-defined objective\n2. Temporary - A project has a definite beginning and a definite end.\n3. Progressive elabora

In [28]:
rag_retriever.retrieve("Advantages of Project Management")

Retrieving documents for query: 'Advantages of Project Management'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.98it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_19bbf164_8',
  'content': 'Advantages of Project Management\n•Better control of financial, physical, and human resources\n•Improved customer relations\n•Shorter development times\n•Lower costs and improved productivity\n•Higher quality and increased reliability\n•Higher profit margins\n•Better internal coordination\n•Positive impact on meeting strategic goals\n•Higher worker morale',
  'metadata': {'creationdate': '2024-06-01T11:24:39+05:30',
   'author': 'Dell',
   'page': 8,
   'moddate': "D:20240607211157Z00'00",
   'doc_index': 8,
   'total_pages': 31,
   'content_length': 348,
   'producer': 'Microsoft® PowerPoint® 2016',
   'file_type': 'pdf',
   'title': 'Project Management',
   'source': '..\\data\\pdf\\1. Introduction.pdf',
   'source_file': '1. Introduction.pdf',
   'page_label': '9',
   'creator': 'Microsoft® PowerPoint® 2016'},
  'similarity_score': 0.6440700888633728,
  'distance': 0.3559299111366272,
  'rank': 1},
 {'id': 'doc_a05ab915_73',
  'content': 'Proj

### RAG Pipeline- VectorDB To LLM Output Generation

In [30]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

True

In [33]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [39]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, openai/gpt-oss-120b, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [40]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [41]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Advantages of Project Management")

Retrieving documents for query: 'Advantages of Project Management'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_19bbf164_8',
  'content': 'Advantages of Project Management\n•Better control of financial, physical, and human resources\n•Improved customer relations\n•Shorter development times\n•Lower costs and improved productivity\n•Higher quality and increased reliability\n•Higher profit margins\n•Better internal coordination\n•Positive impact on meeting strategic goals\n•Higher worker morale',
  'metadata': {'total_pages': 31,
   'moddate': "D:20240607211157Z00'00",
   'title': 'Project Management',
   'producer': 'Microsoft® PowerPoint® 2016',
   'page': 8,
   'source_file': '1. Introduction.pdf',
   'author': 'Dell',
   'source': '..\\data\\pdf\\1. Introduction.pdf',
   'page_label': '9',
   'file_type': 'pdf',
   'content_length': 348,
   'creationdate': '2024-06-01T11:24:39+05:30',
   'creator': 'Microsoft® PowerPoint® 2016',
   'doc_index': 8},
  'similarity_score': 0.6440700888633728,
  'distance': 0.3559299111366272,
  'rank': 1},
 {'id': 'doc_a05ab915_73',
  'content': 'Proj

### Integration Vectordb Context pipeline With LLM output

In [42]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [43]:
answer=rag_simple("Advantages of Project Management",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Advantages of Project Management'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Advantages of Project Management**

- Better control of financial, physical, and human resources  
- Improved customer relations  
- Shorter development times  
- Lower costs and higher productivity  
- Higher quality and increased reliability  
- Higher profit margins  
- Better internal coordination  
- Positive impact on meeting strategic goals  
- Higher worker morale


### Enhanced RAG Pipeline Features

In [44]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is an Organization?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is an Organization?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: An organization is a group of people with diverse knowledge and skills who work together to achieve a common organizational goal.
Sources: [{'source': '2. Organizations.pdf', 'page': 1, 'score': 0.5217419266700745, 'preview': 'What is an Organization\n•Organization is a group of people having different knowledge and \nskills working together to achieve an organizational goal\n•Projects can’t run in isolation. Projects must be operated in a broad \norganizational environment\n•Project Managers need to take the systems approach ...'}, {'source': '2. Organizations.pdf', 'page': 10, 'score': 0.22626447677612305, 'preview': 'Organizational Structures\n1. Functional Organizations...'}, {'source': '2. Organizations.pdf', 'page': 13, 'score': 0.20983099937438965, 'preview': 'Organizational Structures\n2. Project Organizations...'}]
Confidence: 0.5217419266700745
Context Preview: What is an Organization
•Organization is a group of people having different knowledge and 
skills working to

In [45]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("The Four Frames of Organizations", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'The Four Frames of Organizations'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.45it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
The Four Frames of Organizations
1.Structural Frame
Deals with how the organization is stru

ctured and focuses on different groups roles and 
responsibilities in order to meet the goals and policies set by top management.
Courses of Action:
• Reorganize, implement or clarify policies and procedures
• Develop new information, budgeting or control systems
• Add new organizational units
• Plan Processes

The Four Frames of Organizations
1. Structural Frame
2. Human Resources Frame
3. Political Frame
4. Symbolic Frame

The Four Frames of Organizations
3. Political Frame
Politics in organizations take the form of competition among groups or individuals for power and 
leadership.
Courses of Action:
• Bargain
• Negotiate
• Build Alliance / Network with key roles
• Anticipate Conflicts

Question: The Four Frames of Organizations

Answer:

Final Answer: The Four Frames of Organizations are:

1. **Structural Frame**  
2. **Human Resources Frame**  
3. **Political Frame**  
4. **Symbolic Frame**

Citations:
[1] 2. Organizations.pdf (page 5)
[2] 2. Organizations.pdf (page 4)
[3] 2. Organ